# Task 01: Zero-Shot Topic Classification

Use `knowledgator/gliclass-edge-v3.0` to classify news articles by topic.

You will:
1. Load the model and build multi-label and single-label pipelines
2. Implement `classify_topics(pipeline, texts, labels, threshold)` for batch classification
3. Evaluate single-label accuracy on all articles

In [ ]:
from gliclass import GLiClassModel, ZeroShotClassificationPipeline
from transformers import AutoTokenizer
import torch, json, os

fixtures = os.path.normpath(os.path.join(os.getcwd(), "..", "fixtures", "input"))
with open(os.path.join(fixtures, "news_articles.json")) as f:
    articles = json.load(f)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✓ Loaded {len(articles)} articles, device={device}")

## Part 1: Load Model and Build Pipelines

Load `knowledgator/gliclass-edge-v3.0` and create two pipelines:
- `pipeline_ml`: `classification_type='multi-label'`
- `pipeline_sl`: `classification_type='single-label'`

Remember: `add_prefix_space=True` for the tokenizer.

In [ ]:
# YOUR CODE HERE


In [ ]:
# TEST — pipelines loaded
assert 'pipeline_ml' in vars(), "Create 'pipeline_ml' (multi-label)"
assert 'pipeline_sl' in vars(), "Create 'pipeline_sl' (single-label)"
assert callable(pipeline_ml), "pipeline_ml must be callable"
assert callable(pipeline_sl), "pipeline_sl must be callable"

# Quick smoke test
smoke = pipeline_ml("Apple released a new iPhone.", ["technology", "sports"], threshold=0.0)
assert isinstance(smoke, list) and len(smoke) == 1, "pipeline returns list of lists"
assert isinstance(smoke[0], list), "inner element must be a list"
assert all('label' in r and 'score' in r for r in smoke[0]), "each result needs label+score"

print("✓ Both pipelines loaded and return correct format")

## Part 2: Multi-Label Batch Classification

Implement `classify_topics(pipeline, texts, labels, threshold=0.4) -> list[list[str]]`
that returns a list of predicted topic lists (label names only, not dicts) for each text.

In [ ]:
# YOUR CODE HERE
def classify_topics(pipeline, texts, labels, threshold=0.4):
    """
    Classify multiple texts into topic labels.

    Returns:
        list[list[str]] — predicted label names for each text
    """
    pass


In [ ]:
# TEST — batch classification
TOPIC_LABELS = ["technology", "finance", "sports", "science", "politics",
                "entertainment", "world news", "business", "health", "space"]

texts = [a["text"] for a in articles]
predictions = classify_topics(pipeline_ml, texts, TOPIC_LABELS)

assert isinstance(predictions, list), "classify_topics must return a list"
assert len(predictions) == len(articles), f"Expected {len(articles)} results"
for i, pred in enumerate(predictions):
    assert isinstance(pred, list), f"Result {i} must be a list of strings"
    for label in pred:
        assert isinstance(label, str), f"Result {i}: labels must be strings"

# Spot checks on clear-cut articles
tech_article_idx = 0   # Apple MacBook
sports_article_idx = 2  # Champions League
assert "technology" in predictions[tech_article_idx], \
    f"Article 0 (tech) not classified as technology. Got: {predictions[tech_article_idx]}"
assert "sports" in predictions[sports_article_idx], \
    f"Article 2 (sports) not classified as sports. Got: {predictions[sports_article_idx]}"

print(f"✓ classify_topics works correctly")
for article, pred in zip(articles, predictions):
    print(f"  [{article['domain']:12}] {pred}")

## Part 3: Single-Label Accuracy

Using `pipeline_sl`, classify all articles and compute accuracy against `expected_topics`.
A prediction is correct if the predicted label appears in `article['expected_topics']`.

Store accuracy (0.0–1.0) in `accuracy`.

In [ ]:
# YOUR CODE HERE
TOPIC_LABELS_SL = ["technology", "finance", "sports", "science", "politics",
                   "entertainment", "world news", "business", "health", "space",
                   "disaster"]

# accuracy = ...


In [ ]:
# TEST — accuracy
assert 'accuracy' in vars(), "Store accuracy in variable 'accuracy'"
assert isinstance(accuracy, float), "accuracy must be a float"
assert 0.0 <= accuracy <= 1.0, "accuracy must be between 0 and 1"
assert accuracy >= 0.5, (
    f"Accuracy {accuracy:.0%} is too low — expected at least 50% on these clear-cut articles. "
    "Try more descriptive label names or a lower threshold."
)
print(f"✓ Single-label accuracy: {accuracy:.0%} ({int(accuracy * len(articles))}/{len(articles)})")